In [0]:
# Storage access (move to Secret Scope for real deployments)
# Secrets are fetched at runtime — no hardcoded values.
import os

SECRET_SCOPE = "lakehouse-secrets"


def get_secret(key, default=None):
    """Fetch secret from Databricks Secret Scope, falling back to env var."""
    try:
        return dbutils.secrets.get(scope=SECRET_SCOPE, key=key)
    except Exception:
        return os.getenv(key.upper().replace("-", "_"), default)


storage_account = os.getenv("AZURE_STORAGE_ACCOUNT", "your-storage-account")
storage_key = get_secret("storage-account-key", "")

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    storage_key
)

# Path constants
BRONZE_PATH = f"abfss://bronze@{storage_account}.dfs.core.windows.net/crypto_trades"
SILVER_PATH = f"abfss://silver@{storage_account}.dfs.core.windows.net/crypto_trades"
CHECKPOINT_PATH = f"abfss://checkpoint@{storage_account}.dfs.core.windows.net/silver_checkpoint"

In [0]:
bronze_df = (
    spark.readStream
         .format("delta")
         .load(BRONZE_PATH)
)

In [0]:
from pyspark.sql.functions import col, upper, round, current_timestamp

silver_df = (
    bronze_df
    # Business transformations
    .withColumn("symbol", upper(col("symbol")))
    .withColumn("asset_class", upper(col("asset_class")))
    .withColumn("trade_time", (col("trade_timestamp") / 1000).cast("timestamp"))
    .withColumn("trade_value", round(col("trade_price") * col("trade_quantity"), 2))
    .withColumn("processing_time", current_timestamp())

    # Drop invalid records (bad price/quantity from source)
    .filter((col("trade_price") > 0) & (col("trade_quantity") > 0))

    # Watermark: bounds state size so the engine can expire old entries.
    # Without this, dedup state grows unbounded and eventually OOMs.
    .withWatermark("trade_time", "5 minutes")

    # Dedup: Event Hubs can redeliver messages on consumer restarts.
    # dropDuplicatesWithinWatermark uses the watermark window to keep
    # state finite while still catching duplicates within 5 minutes.
    .dropDuplicatesWithinWatermark(["trade_id"])
)

In [0]:
query = (
    silver_df.writeStream
        .format("delta")
        .outputMode("append")
        .queryName("silver_crypto_trades")
        .option("checkpointLocation", CHECKPOINT_PATH)
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .start(SILVER_PATH)
)

query.awaitTermination()

In [0]:
silver_table = spark.read.format("delta").load(SILVER_PATH)
print(f"✓ Silver record count: {silver_table.count():,}")

display(
    silver_table
        .select("trade_time", "symbol", "trade_price", "trade_quantity", "trade_value")
        .orderBy(col("trade_time").desc())
        .limit(5)
)